# Поиск agr_id и ИНН по списку клиентов с фото

Цель:
1. Зафиксировать список клиентов из фото.
2. Найти их в отчете за апрель 2026 (приоритет: Excel `/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx`, fallback: CSV `datamart_month_acquiring_2026_04.csv`).
3. Получить `agr_id`, `inn`, `company_name`, `contract_number`, `ogrn`.
4. Подсветить спорные случаи (не найдено, несколько кандидатов, низкая уверенность).

In [ ]:
from pathlib import Path
import re
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 240)

## 1) Список клиентов из фото

In [ ]:
clients_from_photo = [
    {'client_name_photo': 'БИКМЕТОВ РАФИК АБДРАХМАНОВИЧ (ИП)', 'contract_number_photo': 'ВД_3-6219/37/20', 'ogrn_photo': '3050229930200032'},
    {'client_name_photo': 'ХУСНУТДИНОВА ОЛЕСЯ ГАЗИНУРОВНА (ИП)', 'contract_number_photo': 'ВД_3-6235/110/16', 'ogrn_photo': '315028000114809'},
    {'client_name_photo': 'ООО "АГРО М"', 'contract_number_photo': 'ВД_3-6210/31/21', 'ogrn_photo': '1060271004067'},
    {'client_name_photo': 'ЛЕ ВАН ХИЕП (ИП)', 'contract_number_photo': 'ВД_3-6221/12/22', 'ogrn_photo': '321028000020282'},
    {'client_name_photo': 'АМИРХАНОВА РУЗИЛЯ РИНАТОВНА (ИП)', 'contract_number_photo': 'ВД_3-6204/13/19', 'ogrn_photo': '317028000151330'},
    {'client_name_photo': 'СПСК "РОДНИК"', 'contract_number_photo': 'ВД_3-6204/12/20', 'ogrn_photo': '1180280059827'},
    {'client_name_photo': 'ООО "СТРОЙМАСТЕР"', 'contract_number_photo': 'ВД_3-6228/90/17', 'ogrn_photo': '1057401005683'},
    {'client_name_photo': 'ХАНЖИНА ЮЛИЯ ПАВЛОВНА (ИП)', 'contract_number_photo': 'ВД_3-6219/24/21', 'ogrn_photo': '311028006900066'},
    {'client_name_photo': 'РАМАЗАНОВ РУСТЕМ МАСГУТОВИЧ (ИП)', 'contract_number_photo': 'ВД_3-6229/8/18', 'ogrn_photo': '306024928300022'},
    {'client_name_photo': 'ООО "КИТАЕВКА"', 'contract_number_photo': 'ВД_9Т/3216', 'ogrn_photo': '1034613001444'},
    {'client_name_photo': 'УЛЬЯНОВА ОКСАНА ЮРЬЕВНА (ИП)', 'contract_number_photo': 'ВД_20210303', 'ogrn_photo': '315533200000842'},
    {'client_name_photo': 'МИХЕЛЬКЕВИЧ ЛАРИСА АЛЕКСЕЕВНА (ИП)', 'contract_number_photo': 'ВД_20221810', 'ogrn_photo': '315532100007521'},
    {'client_name_photo': 'СППК "БИФ"', 'contract_number_photo': 'ВД_08122020', 'ogrn_photo': '1175321004187'},
    {'client_name_photo': 'ООО "КОНЖАЛЮКС ПЛЮС"', 'contract_number_photo': 'ВД_20211220', 'ogrn_photo': '1075321008322'},
    {'client_name_photo': 'КРЫЛОВ МИХАИЛ МИХАЙЛОВИЧ (ИП)', 'contract_number_photo': 'ВД_222500/237', 'ogrn_photo': '322547600009711'},
    {'client_name_photo': 'МЕЛЕШКО НАДЕЖДА ВАСИЛЬЕВНА (ИП)', 'contract_number_photo': 'ВД_192500/159', 'ogrn_photo': '307547435100058'},
    {'client_name_photo': 'ЦАПЛЕВ АЛЕКСЕЙ ЮРЬЕВИЧ (ИП)', 'contract_number_photo': 'ВД_222500/248', 'ogrn_photo': '321547600146447'},
    {'client_name_photo': 'ЛУПАШКО АЛЕКСЕЙ НИКОЛАЕВИЧ (ИП)', 'contract_number_photo': 'ВД_202500/200', 'ogrn_photo': '320547600110087'},
    {'client_name_photo': 'ООО "ВЕЛДТЕХМАШ-О"', 'contract_number_photo': 'ВД_01-0509-22', 'ogrn_photo': '1095658014540'},
    {'client_name_photo': 'МУСАГИРОВ ЮРИЙ БИЯРСЛАНОВИЧ (ИП)', 'contract_number_photo': 'ВД_01-0506-20', 'ogrn_photo': '305563510400039'},
    {'client_name_photo': 'ОСИН СЕРГЕЙ АНАТОЛЬЕВИЧ (ИП)', 'contract_number_photo': 'ВД_172ТСП/1500', 'ogrn_photo': '304582931700041'},
    {'client_name_photo': 'ООО "АГРОПРОМФИРМА ЛУЧ"', 'contract_number_photo': 'ВД_073-35-21/264', 'ogrn_photo': '1026601503345'},
    {'client_name_photo': 'ООО "ИКМЭК"', 'contract_number_photo': 'ВД_6700209', 'ogrn_photo': '1021607157110'},
]

photo_df = pd.DataFrame(clients_from_photo)
photo_df.insert(0, 'photo_row_id', np.arange(1, len(photo_df) + 1))
# Приводим префикс договора к корректному виду ЭД_Э (вместо OCR-вариантов ВД_3/ВД_З/ЭД)
photo_df['contract_number_photo'] = (
    photo_df['contract_number_photo']
    .astype(str)
    .str.replace(r'^(?:ВД[_\-]?[3З]|ВД|ЭД[_\-]?Э?)', 'ЭД_Э', regex=True)
)
print(f'Клиентов из фото: {len(photo_df):,}')
photo_df

## 2) Загрузка отчета

In [ ]:
report_sources = [
    ('excel', Path('/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx')),
    ('csv', Path('./datamart_month_acquiring_2026_04.csv')),
    ('csv', Path('/home/jovyan/documents/Equaring/Data/datamart_month_acquiring_2026_04.csv')),
]

report_source = next(((t, p) for t, p in report_sources if p.exists()), None)
if report_source is None:
    raise FileNotFoundError(
        'Не найден отчет за апрель 2026. Ожидался один из путей: '
        '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx '
        'или CSV datamart_month_acquiring_2026_04.csv.'
    )

report_type, report_path = report_source
if report_type == 'excel':
    report_df = pd.read_excel(report_path, dtype='object')
else:
    report_df = pd.read_csv(report_path, dtype='object')

print(f'Загружен отчет ({report_type}): {report_path}')
print(f'Строк: {len(report_df):,}, колонок: {len(report_df.columns)}')
print('Колонки:')
print(sorted(report_df.columns.tolist()))

required_cols = ['company_name', 'contract_number', 'ogrn', 'agr_id', 'inn']
missing_cols = [c for c in required_cols if c not in report_df.columns]
if missing_cols:
    raise ValueError(
        f'В отчете отсутствуют обязательные колонки: {missing_cols}. '
        'Проверь, что используется выгрузка с нужной структурой.'
    )

report_df[required_cols].head(5)

## 3) Поиск клиентов в отчете (`agr_id` и `ИНН`) + подсветка спорных случаев

In [ ]:
def norm_text(v):
    if pd.isna(v):
        return None
    s = str(v).lower().strip()
    s = s.replace('ё', 'е')
    s = re.sub(r'"|«|»|\(|\)', ' ', s)
    s = re.sub(r'[^a-zа-я0-9]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or None


def norm_digits(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v))
    return s or None


def norm_contract(v):
    if pd.isna(v):
        return None
    s = str(v).upper().strip()
    s = s.replace(' ', '')
    s = s.replace('Ё', 'Е')
    s = re.sub(r'[^A-ZА-Я0-9]+', '', s)
    # Нормализуем OCR/ввод по префиксу договора к единому виду ЭД_Э*
    s = re.sub(r'^(?:ВД[3З]?|ЭДЭ?)', 'ЭДЭ', s, count=1)
    return s or None


photo_w = photo_df.copy()
report_w = report_df.copy()

photo_w['name_norm'] = photo_w['client_name_photo'].map(norm_text)
photo_w['contract_norm'] = photo_w['contract_number_photo'].map(norm_contract)
photo_w['ogrn_norm'] = photo_w['ogrn_photo'].map(norm_digits)

report_w['name_norm'] = report_w['company_name'].map(norm_text)
report_w['contract_norm'] = report_w['contract_number'].map(norm_contract)
report_w['ogrn_norm'] = report_w['ogrn'].map(norm_digits)
report_w['inn_norm'] = report_w['inn'].map(norm_digits)
report_w['agr_id_norm'] = report_w['agr_id'].astype(str).str.strip()

results = []
disputed_candidates = []

for _, row in photo_w.iterrows():
    cand = report_w[['company_name', 'contract_number', 'ogrn', 'agr_id', 'inn', 'name_norm', 'contract_norm', 'ogrn_norm']].copy()

    p_name = row['name_norm']
    p_contract = row['contract_norm']
    p_ogrn = row['ogrn_norm']

    cand['name_sim'] = cand['name_norm'].apply(
        lambda x: SequenceMatcher(None, p_name or '', x or '').ratio() if (p_name and x) else 0.0
    )
    cand['name_exact'] = cand['name_norm'].eq(p_name) if p_name else False
    cand['contract_exact'] = cand['contract_norm'].eq(p_contract) if p_contract else False
    cand['ogrn_exact'] = cand['ogrn_norm'].eq(p_ogrn) if p_ogrn else False

    # Интегральный скоринг для ранжирования кандидатов
    cand['score'] = (
        cand['name_sim'] * 100
        + cand['name_exact'].astype(int) * 30
        + cand['contract_exact'].astype(int) * 80
        + cand['ogrn_exact'].astype(int) * 80
    )

    # Ограничиваем пул релевантных кандидатов
    cand_pool = cand[
        cand['contract_exact']
        | cand['ogrn_exact']
        | (cand['name_sim'] >= 0.55)
    ].copy()

    if cand_pool.empty:
        results.append({
            'photo_row_id': row['photo_row_id'],
            'client_name_photo': row['client_name_photo'],
            'contract_number_photo': row['contract_number_photo'],
            'ogrn_photo': row['ogrn_photo'],
            'match_status': 'not_found',
            'match_reason': 'Нет кандидатов по contract/ogrn и name_sim<0.55',
            'is_disputed': True,
            'matched_company_name': None,
            'matched_contract_number': None,
            'matched_ogrn': None,
            'matched_agr_id': None,
            'matched_inn': None,
            'match_score': None,
            'name_similarity': None,
        })
        continue

    cand_pool = cand_pool.sort_values(['score', 'name_sim'], ascending=False).reset_index(drop=True)
    best = cand_pool.iloc[0]

    top_score = float(best['score'])
    near_top = cand_pool[cand_pool['score'] >= (top_score - 5)].copy()
    near_top_unique_pairs = near_top[['agr_id', 'inn']].drop_duplicates()

    has_id_exact = bool(best['contract_exact'] or best['ogrn_exact'])
    is_ambiguous_multi = len(near_top_unique_pairs) > 1

    if has_id_exact and not is_ambiguous_multi:
        status = 'matched_by_identifier'
        reason = 'Есть точное совпадение по contract и/или ogrn'
        disputed = False
    elif best['name_sim'] >= 0.90 and not is_ambiguous_multi:
        status = 'matched_by_name_high'
        reason = 'Высокое сходство имени (>=0.90)'
        disputed = False
    elif is_ambiguous_multi:
        status = 'ambiguous_multi_candidates'
        reason = f'Несколько кандидатов с близким score (>= {top_score - 5:.1f})'
        disputed = True
    else:
        status = 'ambiguous_low_confidence'
        reason = f'Нет точного id-матча, name_sim={best["name_sim"]:.3f}'
        disputed = True

    results.append({
        'photo_row_id': row['photo_row_id'],
        'client_name_photo': row['client_name_photo'],
        'contract_number_photo': row['contract_number_photo'],
        'ogrn_photo': row['ogrn_photo'],
        'match_status': status,
        'match_reason': reason,
        'is_disputed': disputed,
        'matched_company_name': best['company_name'],
        'matched_contract_number': best['contract_number'],
        'matched_ogrn': best['ogrn'],
        'matched_agr_id': best['agr_id'],
        'matched_inn': best['inn'],
        'match_score': round(float(best['score']), 2),
        'name_similarity': round(float(best['name_sim']), 4),
    })

    if disputed:
        top_k = cand_pool.head(5).copy()
        top_k.insert(0, 'photo_row_id', row['photo_row_id'])
        top_k.insert(1, 'client_name_photo', row['client_name_photo'])
        top_k.insert(2, 'contract_number_photo', row['contract_number_photo'])
        top_k.insert(3, 'ogrn_photo', row['ogrn_photo'])
        disputed_candidates.append(top_k)

result_df = pd.DataFrame(results).sort_values('photo_row_id').reset_index(drop=True)
disputed_df = result_df[result_df['is_disputed']].copy().reset_index(drop=True)
disputed_candidates_df = (
    pd.concat(disputed_candidates, ignore_index=True)
    if disputed_candidates else pd.DataFrame()
)

print('Итог поиска в отчете:')
print(result_df['match_status'].value_counts(dropna=False))
print(f"Спорных случаев: {len(disputed_df):,} из {len(result_df):,}")

print('\nОсновной результат (agr_id + inn):')
result_df

## 4) Спорные случаи (подсвечено для ручной проверки)

In [ ]:
if disputed_df.empty:
    print('Спорных случаев не найдено.')
else:
    print('Спорные строки (требуют ручной валидации):')
    display(disputed_df)

if disputed_candidates_df.empty:
    print('Дополнительные кандидаты для спорных кейсов отсутствуют.')
else:
    print('TOP-5 кандидатов по каждому спорному случаю:')
    cols = [
        'photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn_photo',
        'company_name', 'contract_number', 'ogrn', 'agr_id', 'inn',
        'contract_exact', 'ogrn_exact', 'name_exact', 'name_sim', 'score'
    ]
    display(disputed_candidates_df[cols])

## 5) Новый список клиентов с фото (18 июня)

Ниже отдельный список по новым фото. Это отдельная проверка, не смешанная с предыдущим набором.

In [ ]:
clients_from_photo_2026_06_18 = [
    {'client_name_photo': 'БИКМЕТОВА САЗИДА АСХАТОВНА (ИП)', 'contract_number_photo': 'ЭД266219/00028', 'ogrn_photo': '325028000279052'},
    {'client_name_photo': 'ЛАРИН АЛЕКСАНДР НИКОЛАЕВИЧ (ИП ГКФХ)', 'contract_number_photo': 'ЭД266213/00034', 'ogrn_photo': '304025512800036'},
    {'client_name_photo': 'БАГАУТДИНОВ РАБИЛ ЗИЯФОВИЧ (ИП)', 'contract_number_photo': 'ЭД266206/00045', 'ogrn_photo': '326028000007223'},
    {'client_name_photo': 'ЛИ ЖЭНЬШУНЬ (ИП)', 'contract_number_photo': 'ЭД266235/00033', 'ogrn_photo': '320028000097857'},
    {'client_name_photo': 'ООО "НЬЮРИС"', 'contract_number_photo': 'ВД_300', 'ogrn_photo': '1149102120441'},
    {'client_name_photo': 'ФРАНТЕНКО ВИКТОРИЯ КОНСТАНТИНОВНА (ИП)', 'contract_number_photo': 'ЭД256615/00027', 'ogrn_photo': '323385000127952'},
    {'client_name_photo': 'ООО "КРОСС+"', 'contract_number_photo': 'ВД_000098', 'ogrn_photo': '1053811141230'},
    {'client_name_photo': 'ООО "САНИВА"', 'contract_number_photo': 'ЭД263226/00020', 'ogrn_photo': '1114613000490'},
    {'client_name_photo': 'ООО "СВТ"', 'contract_number_photo': 'ЭД250800/00022', 'ogrn_photo': '1095321000070'},
    {'client_name_photo': 'ООО "КУПЕЦ"', 'contract_number_photo': 'ЭД262553/00005', 'ogrn_photo': '1105543001520'},
    {'client_name_photo': 'КИЯН МАРИНА НИКОЛАЕВНА (ИП)', 'contract_number_photo': 'ЭД262504/00044', 'ogrn_photo': '326547600049591'},
    {'client_name_photo': 'ЛАНГ ЭДУАРД ЭДУАРДОВИЧ (ИП)', 'contract_number_photo': 'ЭД262500/00043', 'ogrn_photo': '322547600045721'},
    {'client_name_photo': 'МУШЕНКО МАКСИМ АНАТОЛЬЕВИЧ (ИП)', 'contract_number_photo': 'ЭД252500/00007', 'ogrn_photo': '323547600047523'},
    {'client_name_photo': 'АЛЛАЗОВА СУДАБА ИЛЬЯС КЫЗЫ (ИП)', 'contract_number_photo': 'ЭД252500/00010', 'ogrn_photo': '324246800091492'},
    {'client_name_photo': 'ООО "ПРИВОЛЬНОЕ"', 'contract_number_photo': 'ЭД260519/00077', 'ogrn_photo': '1115658002020'},
    {'client_name_photo': 'МИНАЗКИРОВ КИРИЛЛ ЭДУАРДОВИЧ (ИП)', 'contract_number_photo': 'ЭД250507/00108', 'ogrn_photo': '325565800030598'},
    {'client_name_photo': 'ООО "САНФОРТ-СЕРВИС"', 'contract_number_photo': 'ЭД261500/00015', 'ogrn_photo': '1025801369076'},
    {'client_name_photo': 'ТУГАЕВА ЮЛИЯ ВАЛЕРЬЕВНА (ИП)', 'contract_number_photo': 'ЭД261500/00016', 'ogrn_photo': '307583611500025'},
    {'client_name_photo': 'ШАБАЛИН ДМИТРИЙ НИКОЛАЕВИЧ (ИП ГКФХ)', 'contract_number_photo': 'ЭД267605/00036', 'ogrn_photo': '325595800099432'},
    {'client_name_photo': 'ООО "ЛИОНА"', 'contract_number_photo': 'ЭД267605/00043', 'ogrn_photo': '1225900023315'},
    {'client_name_photo': 'СЕЛЕЗНЕВА МАРГАРИТА ФАНИЛЬЕВНА (ИП)', 'contract_number_photo': 'ЭД267309/00035', 'ogrn_photo': '326965800023753'},
    {'client_name_photo': 'ПОПОВ АЛЕКСАНДР ВЛАДИМИРОВИЧ (ИП ГКФХ)', 'contract_number_photo': 'ЭД267300/00047', 'ogrn_photo': '319665800113563'},
    {'client_name_photo': 'НЕСТЕРОВА МАРИНА АНДРЕЕВНА (ИП)', 'contract_number_photo': 'ЭД267306/00038', 'ogrn_photo': '322665800105220'},
    {'client_name_photo': 'ЗИННАТУЛЛИНА ГУЗАЛЬ ВАКЫФОВНА (ИП ГКФХ)', 'contract_number_photo': 'ЭД266700/00045', 'ogrn_photo': '312167520800136'},
    {'client_name_photo': 'ГИЗЕТДИНОВ ИЛЬНУР АЛЬБЕРТОВИЧ (ИП)', 'contract_number_photo': 'ЭД266721/00037', 'ogrn_photo': '326169000072080'},
    {'client_name_photo': 'СИБГАТУЛЛИНА АЛИЯ ИРЕКОВНА (ИП)', 'contract_number_photo': 'ЭД256705/00004', 'ogrn_photo': '320169000141554'},
    {'client_name_photo': 'ООО "АВТОДЕТАЛИ"', 'contract_number_photo': 'ЭД266716/00040', 'ogrn_photo': '1131651000250'},
    {'client_name_photo': 'АШИМЖОНОВ РАСУЛЖОН АРАБЖОНОВИЧ (ИП)', 'contract_number_photo': 'ЭД256705/00005', 'ogrn_photo': '313169000166854'},
]

photo_2026_06_18_df = pd.DataFrame(clients_from_photo_2026_06_18)
photo_2026_06_18_df.insert(0, 'photo_row_id', np.arange(1, len(photo_2026_06_18_df) + 1))
# Приводим префикс договора к корректному виду ЭД_Э (вместо OCR-вариантов ВД_3/ВД_З/ЭД)
photo_2026_06_18_df['contract_number_photo'] = (
    photo_2026_06_18_df['contract_number_photo']
    .astype(str)
    .str.replace(r'^(?:ВД[_\-]?[3З]|ВД|ЭД[_\-]?Э?)', 'ЭД_Э', regex=True)
)
print(f'Клиентов из новых фото: {len(photo_2026_06_18_df):,}')
photo_2026_06_18_df

## 6) Поиск `agr_id` и `ИНН` в отчете + спорные случаи (новый список)

In [ ]:
from pathlib import Path
import re
from difflib import SequenceMatcher

import numpy as np
import pandas as pd


def norm_text_local(v):
    if pd.isna(v):
        return None
    s = str(v).lower().strip()
    s = s.replace('ё', 'е')
    s = re.sub(r'"|«|»|\(|\)', ' ', s)
    s = re.sub(r'[^a-zа-я0-9]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or None


def norm_digits_local(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v))
    return s or None


def norm_contract_local(v):
    if pd.isna(v):
        return None
    s = str(v).upper().strip()
    s = s.replace(' ', '')
    s = s.replace('Ё', 'Е')
    s = re.sub(r'[^A-ZА-Я0-9]+', '', s)
    # Нормализуем OCR/ввод по префиксу договора к единому виду ЭД_Э*
    s = re.sub(r'^(?:ВД[3З]?|ЭДЭ?)', 'ЭДЭ', s, count=1)
    return s or None


# Подгружаем отчет, если он еще не загружен в текущем ядре
if 'report_df' not in globals() or report_df is None or report_df.empty:
    report_sources = [
        ('excel', Path('/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx')),
        ('csv', Path('./datamart_month_acquiring_2026_04.csv')),
        ('csv', Path('/home/jovyan/documents/Equaring/Data/datamart_month_acquiring_2026_04.csv')),
    ]
    report_source = next(((t, p) for t, p in report_sources if p.exists()), None)
    if report_source is None:
        raise FileNotFoundError(
            'Не найден отчет за апрель 2026. Ожидался один из путей: '
            '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx '
            'или CSV datamart_month_acquiring_2026_04.csv.'
        )

    report_type, report_path = report_source
    if report_type == 'excel':
        report_df = pd.read_excel(report_path, dtype='object')
    else:
        report_df = pd.read_csv(report_path, dtype='object')

    print(f'Загружен отчет ({report_type}): {report_path}')

required_cols = ['company_name', 'contract_number', 'ogrn', 'agr_id', 'inn']
missing_cols = [c for c in required_cols if c not in report_df.columns]
if missing_cols:
    raise ValueError(f'В отчете отсутствуют обязательные колонки: {missing_cols}')

photo_w = photo_2026_06_18_df.copy()
report_w = report_df.copy()

photo_w['name_norm'] = photo_w['client_name_photo'].map(norm_text_local)
photo_w['contract_norm'] = photo_w['contract_number_photo'].map(norm_contract_local)
photo_w['ogrn_norm'] = photo_w['ogrn_photo'].map(norm_digits_local)

report_w['name_norm'] = report_w['company_name'].map(norm_text_local)
report_w['contract_norm'] = report_w['contract_number'].map(norm_contract_local)
report_w['ogrn_norm'] = report_w['ogrn'].map(norm_digits_local)
report_w['inn_norm'] = report_w['inn'].map(norm_digits_local)
report_w['agr_id_norm'] = report_w['agr_id'].astype(str).str.strip()

results = []
disputed_candidates = []

for _, row in photo_w.iterrows():
    cand = report_w[['company_name', 'contract_number', 'ogrn', 'agr_id', 'inn', 'name_norm', 'contract_norm', 'ogrn_norm']].copy()

    p_name = row['name_norm']
    p_contract = row['contract_norm']
    p_ogrn = row['ogrn_norm']

    cand['name_sim'] = cand['name_norm'].apply(
        lambda x: SequenceMatcher(None, p_name or '', x or '').ratio() if (p_name and x) else 0.0
    )
    cand['name_exact'] = cand['name_norm'].eq(p_name) if p_name else False
    cand['contract_exact'] = cand['contract_norm'].eq(p_contract) if p_contract else False
    cand['ogrn_exact'] = cand['ogrn_norm'].eq(p_ogrn) if p_ogrn else False

    cand['score'] = (
        cand['name_sim'] * 100
        + cand['name_exact'].astype(int) * 30
        + cand['contract_exact'].astype(int) * 80
        + cand['ogrn_exact'].astype(int) * 80
    )

    cand_pool = cand[
        cand['contract_exact']
        | cand['ogrn_exact']
        | (cand['name_sim'] >= 0.55)
    ].copy()

    if cand_pool.empty:
        results.append({
            'photo_row_id': row['photo_row_id'],
            'client_name_photo': row['client_name_photo'],
            'contract_number_photo': row['contract_number_photo'],
            'ogrn_photo': row['ogrn_photo'],
            'match_status': 'not_found',
            'match_reason': 'Нет кандидатов по contract/ogrn и name_sim<0.55',
            'is_disputed': True,
            'matched_company_name': None,
            'matched_contract_number': None,
            'matched_ogrn': None,
            'matched_agr_id': None,
            'matched_inn': None,
            'match_score': None,
            'name_similarity': None,
        })
        continue

    cand_pool = cand_pool.sort_values(['score', 'name_sim'], ascending=False).reset_index(drop=True)
    best = cand_pool.iloc[0]

    top_score = float(best['score'])
    near_top = cand_pool[cand_pool['score'] >= (top_score - 5)].copy()
    near_top_unique_pairs = near_top[['agr_id', 'inn']].drop_duplicates()

    has_id_exact = bool(best['contract_exact'] or best['ogrn_exact'])
    is_ambiguous_multi = len(near_top_unique_pairs) > 1

    if has_id_exact and not is_ambiguous_multi:
        status = 'matched_by_identifier'
        reason = 'Есть точное совпадение по contract и/или ogrn'
        disputed = False
    elif best['name_sim'] >= 0.90 and not is_ambiguous_multi:
        status = 'matched_by_name_high'
        reason = 'Высокое сходство имени (>=0.90)'
        disputed = False
    elif is_ambiguous_multi:
        status = 'ambiguous_multi_candidates'
        reason = f'Несколько кандидатов с близким score (>= {top_score - 5:.1f})'
        disputed = True
    else:
        status = 'ambiguous_low_confidence'
        reason = f'Нет точного id-матча, name_sim={best["name_sim"]:.3f}'
        disputed = True

    results.append({
        'photo_row_id': row['photo_row_id'],
        'client_name_photo': row['client_name_photo'],
        'contract_number_photo': row['contract_number_photo'],
        'ogrn_photo': row['ogrn_photo'],
        'match_status': status,
        'match_reason': reason,
        'is_disputed': disputed,
        'matched_company_name': best['company_name'],
        'matched_contract_number': best['contract_number'],
        'matched_ogrn': best['ogrn'],
        'matched_agr_id': best['agr_id'],
        'matched_inn': best['inn'],
        'match_score': round(float(best['score']), 2),
        'name_similarity': round(float(best['name_sim']), 4),
    })

    if disputed:
        top_k = cand_pool.head(5).copy()
        top_k.insert(0, 'photo_row_id', row['photo_row_id'])
        top_k.insert(1, 'client_name_photo', row['client_name_photo'])
        top_k.insert(2, 'contract_number_photo', row['contract_number_photo'])
        top_k.insert(3, 'ogrn_photo', row['ogrn_photo'])
        disputed_candidates.append(top_k)

result_2026_06_18_df = pd.DataFrame(results).sort_values('photo_row_id').reset_index(drop=True)
disputed_2026_06_18_df = result_2026_06_18_df[result_2026_06_18_df['is_disputed']].copy().reset_index(drop=True)
disputed_candidates_2026_06_18_df = (
    pd.concat(disputed_candidates, ignore_index=True)
    if disputed_candidates else pd.DataFrame()
)

print('Итог поиска в отчете (новый список):')
print(result_2026_06_18_df['match_status'].value_counts(dropna=False))
print(f"Спорных случаев: {len(disputed_2026_06_18_df):,} из {len(result_2026_06_18_df):,}")

print('\nОсновной результат (agr_id + inn):')
display(result_2026_06_18_df)

if disputed_2026_06_18_df.empty:
    print('Спорных случаев не найдено.')
else:
    print('Спорные строки (требуют ручной валидации):')
    display(disputed_2026_06_18_df)

if disputed_candidates_2026_06_18_df.empty:
    print('Дополнительные кандидаты для спорных кейсов отсутствуют.')
else:
    print('TOP-5 кандидатов по каждому спорному случаю:')
    cols = [
        'photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn_photo',
        'company_name', 'contract_number', 'ogrn', 'agr_id', 'inn',
        'contract_exact', 'ogrn_exact', 'name_exact', 'name_sim', 'score'
    ]
    display(disputed_candidates_2026_06_18_df[cols])

## 7) Подтяжка `tariff_name` из Озера по найденным `agr_id` (новый список)

In [ ]:
# Берем только найденные agr_id из нового списка (включая спорные, если agr_id определился)
matched_agr_ids_2026_06_18 = (
    result_2026_06_18_df['matched_agr_id']
    .dropna()
    .astype(str)
    .str.strip()
)
matched_agr_ids_2026_06_18 = sorted({x for x in matched_agr_ids_2026_06_18 if x})

if not matched_agr_ids_2026_06_18:
    print('Нет найденных agr_id для подтяжки tariff_name.')
else:
    try:
        from rail_connectors.connection import connect
    except Exception as e:
        raise ImportError(
            'Не удалось импортировать rail_connectors.connection.connect. '
            'Проверь окружение с доступом к Озеру.'
        ) from e

    if 'imp' not in globals() or imp is None:
        imp = connect('impala')

    agr_ids_sql = ', '.join([f"'{x}'" for x in matched_agr_ids_2026_06_18])

    sql_tariff_name = f"""
    WITH m AS (
        SELECT
            CAST(c_contract AS STRING) AS agr_id,
            c_tariff_plan,
            d_start,
            d_end,
            CAST(c_row_status AS STRING) AS row_status
        FROM ods.scd1_z_r2_ip_merchants
        WHERE CAST(c_contract AS STRING) IN ({agr_ids_sql})
    )
    SELECT
        m.agr_id,
        m.c_tariff_plan,
        tp.c_tariff_name AS tariff_name,
        m.d_start,
        m.d_end,
        m.row_status
    FROM m
    LEFT JOIN ods.scd1_z_r2_tariff_plan tp
        ON m.c_tariff_plan = tp.c_tariff
    """

    tariff_2026_06_18_df = imp.sql(sql_tariff_name)
    tariff_2026_06_18_df['agr_id'] = tariff_2026_06_18_df['agr_id'].astype(str).str.strip()

    tariff_best_2026_06_18_df = (
        tariff_2026_06_18_df
        .sort_values(['agr_id', 'd_start', 'd_end'], ascending=[True, False, False])
        .drop_duplicates(subset=['agr_id'], keep='first')
        [['agr_id', 'tariff_name', 'c_tariff_plan', 'row_status', 'd_start', 'd_end']]
        .rename(columns={
            'agr_id': 'matched_agr_id',
            'tariff_name': 'tariff_name_lake',
            'c_tariff_plan': 'tariff_plan_lake',
            'row_status': 'merchant_row_status_lake',
            'd_start': 'merchant_d_start_lake',
            'd_end': 'merchant_d_end_lake',
        })
    )

    result_2026_06_18_with_tariff_df = result_2026_06_18_df.merge(
        tariff_best_2026_06_18_df,
        on='matched_agr_id',
        how='left'
    )

    print(f'agr_id для подтяжки тарифа: {len(matched_agr_ids_2026_06_18):,}')
    print(f'Строк из Озера (ip_merchants): {len(tariff_2026_06_18_df):,}')

    print('\nИтог (новый список клиентов + tariff_name из Озера):')
    display(result_2026_06_18_with_tariff_df)

    print('\nТолько спорные кейсы с тарифом (если есть):')
    display(result_2026_06_18_with_tariff_df[result_2026_06_18_with_tariff_df['is_disputed']].copy())